In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col

# Inicializa a sessão Spark
spark = SparkSession.builder.getOrCreate()

caminho_csv = "/Volumes/grao_direto/default/dados_origem/grain_logistic_shipping  (1) (3) (1).csv"

df_bronze = (
    spark.read
    .option("header", True)
    .option("delimiter", ";")
    .option("includeMetadata", "true")  
    .csv(caminho_csv)
    .withColumn("dt_processamento", current_timestamp()) # Adiciona coluna com timestamp de processamento para rastreabilidade
    .withColumn("file_path", col("_metadata.file_path")) # Adiciona coluna com o caminho do arquivo de origem, útil para auditoria
)

# Dados em formato Delta na camada Bronze
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("grao_direto.default.tabela_bronze_grain_logistic")

print("Camada Bronze finalizada com sucesso!")
